# 7-Distill

对应 `trainer/train_distillation.py`。

- **黑盒蒸馏**：学教师的最终答案，本质还是 SFT。主线 `sft_t2t` 里已经混了大量 Qwen / R1 风格数据。
- **白盒蒸馏**：再拟合教师的 token 分布。

$$\mathcal{L} = \alpha\,\mathrm{CE} + (1-\alpha)\,T^{2}\,\mathrm{KL}(p_t^{T}\parallel p_s^{T})$$


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)
def tiny_model(use_moe=False):
    cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=use_moe)
    return MiniMindForCausalLM(cfg).to(device), cfg

import torch.nn.functional as F


In [ ]:
def distillation_loss(student_logits, teacher_logits, temperature=2.0):
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1).detach()
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    return (temperature ** 2) * F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")


In [ ]:
student, _ = tiny_model()
teacher, teacher_cfg = tiny_model()
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

ds = SFTDataset("./toydata/sft_data.jsonl", tokenizer, max_length=96)
loader = DataLoader(ds, batch_size=2)
optimizer = optim.AdamW(student.parameters(), lr=5e-4)
alpha, temperature = 0.5, 2.0
student.train()

for step, (input_ids, labels) in enumerate(loader, start=1):
    input_ids, labels = input_ids.to(device), labels.to(device)
    mask = (labels[..., 1:] != -100)
    s_logits = student(input_ids).logits[..., :-1, :]
    with torch.no_grad():
        t_logits = teacher(input_ids).logits[..., :-1, :]
    ce = F.cross_entropy(s_logits.reshape(-1, s_logits.size(-1)), labels[..., 1:].reshape(-1), ignore_index=-100)
    kl = distillation_loss(s_logits[mask], t_logits[mask], temperature)
    loss = alpha * ce + (1 - alpha) * kl
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(f"step {step} loss={float(loss):.4f} ce={float(ce):.4f} kl={float(kl):.4f}")


完整训练：`cd trainer && python train_distillation.py`。教师/学生可以是不同 `hidden_size` 的 MiniMind。
